# Materialize Gold Fact Table

NYC TLC Yellow Taxi 

## Purpose
This notebook materializes a **Gold fact Delta table** designed for a blog series about **semantic model memory optimization** in Microsoft Fabric.

The table intentionally includes “model-hostile” patterns such as:
- Audit / lineage columns
- High-cardinality identifiers
- Duplicate attributes
- String versions of datetimes and numbers
- Floating-point columns with unnecessary precision

This allows you to demonstrate the memory impact of common modeling choices and the benefits of optimization strategies.

---

## Inputs
- Curated table: `tlc_yellow_tripdata_unified`
- Dimensions:
  - `dim_vendor`, `dim_ratecode`, `dim_payment_type`, `dim_taxi_zone`, `dim_date`, `dim_time_of_day` (optional)

---

## Output
- Gold Delta table: `fact_yellow_taxi_gold`
  - Partitioned by `year`, `month`

You may optionally create a **T-SQL view** on top of the Gold table to control:
- Which columns to expose to Power BI
- Naming conventions
- Filters and row-level shaping per blog post

---


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

## Configuration

Set table names and choose which “model-hostile” columns to include.

In [ ]:
# ----------------------------
# Configuration
# ----------------------------
SILVER_TABLE = "tlc_yellow_tripdata_unified"
GOLD_TABLE   = "fact_yellow_taxi"

# Pipeline metadata
PIPELINE_RUN_ID = "00db5215-2277-4f34-83c1-ae9b9af31b97"   # replace as you like (or pass in from orchestration)

# Include switches (kept as flags for easy blog-post variants)
INCLUDE_AUDIT_COLUMNS = True
INCLUDE_HIGH_CARDINALITY_IDS = True
INCLUDE_DUPLICATE_RAW_KEYS = True
INCLUDE_STRING_VERSIONS = True
INCLUDE_EXTRA_PRECISION_NUMERICS = True
INCLUDE_HASH_COLUMNS = True

# Output options
PARTITION_BY_YEAR_MONTH = True
DO_OPTIMIZE = False          # set True if you want to run OPTIMIZE at the end
ZORDER_COLS = ["year", "month", "PULocationID_clean", "DOLocationID_clean"]

## Helpers

Utility functions for deterministic UUID-like IDs, clean keys, and derived columns.

In [ ]:
def sha_to_uuid(hex_col):
    # Turn a 32-hex string into UUID shape 8-4-4-4-12
    return F.concat_ws(
        "-",
        F.substring(hex_col, 1, 8),
        F.substring(hex_col, 9, 4),
        F.substring(hex_col, 13, 4),
        F.substring(hex_col, 17, 4),
        F.substring(hex_col, 21, 12),
    )

def time_key_from_ts(ts_col):
    return (F.hour(ts_col) * 100 + F.minute(ts_col)).cast("int")

## Load Silver and unify datetimes

Unifies pickup/dropoff timestamps across historical eras and derives calendar keys.

In [ ]:
df = spark.table(SILVER_TABLE)

# Capture lineage BEFORE any joins (input_file_name() requires single source)
df = df.withColumn("source_file_path", F.input_file_name())

pickup_dt  = F.coalesce(F.col("tpep_pickup_datetime"),  F.col("pickup_datetime_legacy"))
dropoff_dt = F.coalesce(F.col("tpep_dropoff_datetime"), F.col("dropoff_datetime_legacy"))

df = (
    df
    .withColumn("pickup_datetime", pickup_dt)
    .withColumn("dropoff_datetime", dropoff_dt)
)

## Clean dimension keys

Creates `*_clean` keys that are guaranteed to join to dimensions (Unknown = -1).
Raw keys are preserved (optional) for auditing and “bad model” demonstrations.

In [ ]:
# Load dimensions for membership checks (zones)
zones = spark.table("dim_taxi_zone").select(F.col("LocationID").alias("zone_id")).distinct()

df = (
    df
    # VendorID clean (dictionary-valid: 1,2,6,7; else -1)
    .withColumn("VendorID_clean", F.when(F.col("VendorID").isin([1,2,6,7]), F.col("VendorID")).otherwise(F.lit(-1)).cast("int"))

    # RatecodeID clean (valid 1..6 else -1)
    .withColumn("RatecodeID_clean",
                F.when(F.col("RatecodeID").isin([1,2,3,4,5,6]), F.col("RatecodeID")).otherwise(F.lit(-1)).cast("int"))

    # payment_type clean (valid 1..6 else -1)
    .withColumn("payment_type_clean",
                F.when(F.col("payment_type").isin([1,2,3,4,5,6]), F.col("payment_type")).otherwise(F.lit(-1)).cast("int"))
)

# Zone conformance using membership check
df = (
    df
    .join(zones, df.PULocationID == zones.zone_id, "left")
    .withColumn("PULocationID_clean", F.when(F.col("zone_id").isNotNull(), F.col("PULocationID")).otherwise(F.lit(-1)).cast("int"))
    .drop("zone_id")
)

df = (
    df
    .join(zones, df.DOLocationID == zones.zone_id, "left")
    .withColumn("DOLocationID_clean", F.when(F.col("zone_id").isNotNull(), F.col("DOLocationID")).otherwise(F.lit(-1)).cast("int"))
    .drop("zone_id")
)

## Date and time keys

Derives:
- `pickup_date_key`, `dropoff_date_key` (YYYYMMDD)
- `pickup_time_key`, `dropoff_time_key` (HHMM)
- High-cardinality numeric timestamp forms (epoch seconds)

In [ ]:
df = (
    df
    .withColumn("pickup_date_key",  F.date_format("pickup_datetime", "yyyyMMdd").cast("int"))
    .withColumn("dropoff_date_key", F.date_format("dropoff_datetime", "yyyyMMdd").cast("int"))
    .withColumn("pickup_time_key",  time_key_from_ts(F.col("pickup_datetime")))
    .withColumn("dropoff_time_key", time_key_from_ts(F.col("dropoff_datetime")))
    .withColumn("pickup_epoch_seconds",  F.unix_timestamp("pickup_datetime").cast("long"))
    .withColumn("dropoff_epoch_seconds", F.unix_timestamp("dropoff_datetime").cast("long"))
)

## High-cardinality identifiers

Adds deterministic UUID-like identifiers to demonstrate semantic model memory impact.

- `trip_surrogate_id`: high-cardinality for all rows  
- `payment_identifier`: high-cardinality only for credit card rows (`payment_type_clean = 1`)

In [ ]:
if INCLUDE_HIGH_CARDINALITY_IDS:
    seed = F.concat_ws(
        "||",
        F.col("pickup_datetime").cast("string"),
        F.col("dropoff_datetime").cast("string"),
        F.col("VendorID").cast("string"),
        F.col("PULocationID_clean").cast("string"),
        F.col("DOLocationID_clean").cast("string"),
        F.col("total_amount").cast("string"),
        F.col("trip_distance").cast("string"),
        F.col("passenger_count").cast("string"),
    )

    hash32_trip = F.substring(F.sha2(seed, 256), 1, 32)
    trip_uuid = sha_to_uuid(hash32_trip)

    # payment identifier can use the same base seed but with a different salt to avoid identical ids
    seed_pay = F.concat_ws("||", F.lit("pay"), seed)
    hash32_pay = F.substring(F.sha2(seed_pay, 256), 1, 32)
    pay_uuid = sha_to_uuid(hash32_pay)

    df = (
        df
        .withColumn("trip_surrogate_id", trip_uuid)
        .withColumn("payment_identifier",
                    F.when(F.col("payment_type_clean") == 1, pay_uuid).otherwise(F.lit(None).cast("string")))
    )

## Audit / lineage / hash columns

Adds columns that are common in engineering layers but expensive or rarely useful in a semantic model.

In [ ]:
if INCLUDE_AUDIT_COLUMNS:
    df = (
        df
        .withColumn("ingest_utc_ts", F.current_timestamp())
        .withColumn("pipeline_run_id", F.lit(PIPELINE_RUN_ID))
    )

if INCLUDE_HASH_COLUMNS:
    # Hash a broad set of values to create a very high-cardinality string column
    hash_seed = F.concat_ws(
        "||",
        F.col("trip_surrogate_id") if "trip_surrogate_id" in df.columns else F.lit(""),
        F.col("VendorID").cast("string"),
        F.col("RatecodeID").cast("string"),
        F.col("payment_type").cast("string"),
        F.col("PULocationID").cast("string"),
        F.col("DOLocationID").cast("string"),
        F.col("fare_amount").cast("string"),
        F.col("tip_amount").cast("string"),
        F.col("total_amount").cast("string"),
        F.col("pickup_datetime").cast("string"),
        F.col("dropoff_datetime").cast("string"),
    )
    df = df.withColumn("record_sha2", F.sha2(hash_seed, 256))

## String versions (intentionally expensive)

Adds string representations of common fields to demonstrate how strings inflate model memory.

Examples:
- Datetime as text
- Amounts as text

In [ ]:
if INCLUDE_STRING_VERSIONS:
    df = (
        df
        .withColumn("pickup_datetime_text",  F.date_format("pickup_datetime", "yyyy-MM-dd HH:mm:ss"))
        .withColumn("dropoff_datetime_text", F.date_format("dropoff_datetime", "yyyy-MM-dd HH:mm:ss"))
        .withColumn("pickup_hour_text", F.format_string("%02d", F.hour("pickup_datetime")))
        .withColumn("fare_amount_text", F.col("fare_amount").cast("string"))
        .withColumn("total_amount_text", F.col("total_amount").cast("string"))
    )

## Unnecessary precision numerics

Adds numeric variants that increase distinct values and model memory.

- Scaled doubles (more distinct values)
- Decimal variants (useful for comparison)

In [ ]:
if INCLUDE_EXTRA_PRECISION_NUMERICS:
    df = (
        df
        .withColumn("fare_amount_fp64_scaled", (F.col("fare_amount") * F.lit(1000000.0)).cast("double"))
        .withColumn("total_amount_fp64_scaled", (F.col("total_amount") * F.lit(1000000.0)).cast("double"))
        .withColumn("fare_amount_dec", F.col("fare_amount").cast("decimal(19,4)"))
        .withColumn("total_amount_dec", F.col("total_amount").cast("decimal(19,4)"))
    )

## Duplicate raw keys (optional)

Preserves raw keys alongside cleaned keys to demonstrate the cost of redundant attributes.

In [ ]:
if INCLUDE_DUPLICATE_RAW_KEYS:
    df = (
        df
        .withColumnRenamed("VendorID", "VendorID_raw")
        .withColumnRenamed("RatecodeID", "RatecodeID_raw")
        .withColumnRenamed("payment_type", "payment_type_raw")
        .withColumnRenamed("PULocationID", "PULocationID_raw")
        .withColumnRenamed("DOLocationID", "DOLocationID_raw")
    )

## Final column set

Selects a curated set of columns for the Gold fact table.
This is the table you will create T-SQL views on top of for semantic model scenarios.

In [ ]:
# Base analytical columns
final_cols = [
    # keys
    "pickup_date_key", "dropoff_date_key",
    "pickup_time_key", "dropoff_time_key",
    "VendorID_clean", "RatecodeID_clean", "payment_type_clean",
    "PULocationID_clean", "DOLocationID_clean",

    # core timestamps
    "pickup_datetime", "dropoff_datetime",

    # measures
    "passenger_count", "trip_distance",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "congestion_surcharge", "airport_fee", "cbd_congestion_fee",
    "total_amount",

    # partitions
    "year", "month",
]

# Optionals (included if present)
optional_cols = [
    "trip_surrogate_id", "payment_identifier",
    "pickup_epoch_seconds", "dropoff_epoch_seconds",
    "pickup_datetime_text", "dropoff_datetime_text", "pickup_hour_text",
    "fare_amount_text", "total_amount_text",
    "fare_amount_fp64_scaled", "total_amount_fp64_scaled",
    "fare_amount_dec", "total_amount_dec",
    "ingest_utc_ts", "pipeline_run_id", "source_file_path",
    "record_sha2",
    "vendor_name_raw",
    "pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude",
    "pickup_datetime_legacy", "dropoff_datetime_legacy",
]

for c in optional_cols:
    if c in df.columns:
        final_cols.append(c)

# Raw key duplicates (if enabled)
raw_key_cols = [
    "VendorID_raw", "RatecodeID_raw", "payment_type_raw", "PULocationID_raw", "DOLocationID_raw"
]
for c in raw_key_cols:
    if c in df.columns:
        final_cols.append(c)

df_gold = df.select(*final_cols)

## Write Gold Delta table

Writes `fact_yellow_taxi_gold` as a managed Delta table, partitioned by `year` and `month`.

In [ ]:
writer = (
    df_gold.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
)

if PARTITION_BY_YEAR_MONTH:
    writer = writer.partitionBy("year", "month")

writer.saveAsTable(GOLD_TABLE)

print("Wrote Gold table:", GOLD_TABLE)

## Quick profiling

Produces basic stats useful for your write-up:
- Row count
- Approximate distinct counts for key “memory pain” columns

In [ ]:
t = spark.table(GOLD_TABLE)

print("Rows:", t.count())

pain_cols = [c for c in [
    "trip_surrogate_id",
    "payment_identifier",
    "source_file_path",
    "record_sha2",
    "pickup_datetime_text",
    "fare_amount_text",
    "pickup_epoch_seconds",
    "fare_amount_fp64_scaled",
] if c in t.columns]

exprs = [F.count("*").alias("rows")]
for c in pain_cols:
    exprs.append(F.approx_count_distinct(F.col(c)).alias(f"acd_{c}"))

display(t.agg(*exprs))

## Optional: OPTIMIZE

Optional maintenance step. Consider running this after initial materialization.

In [ ]:
if DO_OPTIMIZE:
    z = ", ".join(ZORDER_COLS)
    spark.sql(f"OPTIMIZE {GOLD_TABLE} ZORDER BY ({z})")
    print("OPTIMIZE completed.")